In [1]:
import copy

# ============================================================
# Lab Assignment 1 - CPU Scheduling Algorithms (FCFS & SJF)
# Course: Operating System Lab ENCA252 | BCA (AI & DS)
# ============================================================


# ─────────────────────────────────────────
# TASK 1: Process Class & Input Handling
# ─────────────────────────────────────────

class Process:
    """Represents a process with scheduling attributes."""
    def __init__(self, pid, arrival_time, burst_time):
        self.pid = pid                  # Process ID
        self.at  = arrival_time         # Arrival Time
        self.bt  = burst_time           # Burst Time
        self.ct  = 0                    # Completion Time (calculated)
        self.tat = 0                    # Turnaround Time (calculated)
        self.wt  = 0                    # Waiting Time    (calculated)

    def __repr__(self):
        return f"Process(PID={self.pid}, AT={self.at}, BT={self.bt})"


def get_processes_from_user():
    """Accept process details interactively from the user."""
    processes = []
    print("\n" + "="*50)
    print("   TASK 1: Process Input")
    print("="*50)
    n = int(input("Enter number of processes (4-5 recommended): "))
    for i in range(n):
        print(f"\n  Process P{i+1}:")
        at = int(input("    Arrival Time : "))
        bt = int(input("    Burst Time   : "))
        processes.append(Process(pid=i+1, arrival_time=at, burst_time=bt))
    return processes


def get_default_processes():
    """Return a default set of 5 processes for demonstration."""
    data = [
        (1, 0, 6),
        (2, 1, 8),
        (3, 2, 7),
        (4, 3, 3),
        (5, 4, 4),
    ]
    return [Process(pid, at, bt) for pid, at, bt in data]


def display_input_table(processes):
    """Display processes in a formatted table (Task 1 output)."""
    print("\n" + "="*40)
    print("   Process Input Table")
    print("="*40)
    print(f"{'PID':<8} {'Arrival Time':<16} {'Burst Time'}")
    print("-"*40)
    for p in processes:
        print(f"P{p.pid:<7} {p.at:<16} {p.bt}")
    print("="*40)


# ─────────────────────────────────────────
# TASK 2: FCFS Scheduling (Non-Preemptive)
# ─────────────────────────────────────────

def fcfs_scheduling(processes):
    """
    First Come First Serve (Non-Preemptive).

    Algorithm:
      1. Sort all processes by Arrival Time.
      2. Execute each in arrival order.
      3. Handle CPU idle gaps if no process has arrived yet.
      4. Compute CT, TAT, WT for each process.

    Returns:
      procs (list)  : Processes with computed metrics.
      gantt (list)  : Gantt chart data as (label, start, end) tuples.
    """
    procs = copy.deepcopy(processes)
    procs.sort(key=lambda p: p.at)   # Step 1: sort by Arrival Time

    current_time = 0
    gantt = []

    for p in procs:
        # Handle CPU idle: if CPU is free but no process has arrived yet
        if current_time < p.at:
            gantt.append(("IDLE", current_time, p.at))
            current_time = p.at

        start = current_time
        current_time += p.bt          # Process executes for its full burst time

        # Calculate scheduling metrics
        p.ct  = current_time           # Completion Time
        p.tat = p.ct - p.at            # Turnaround Time = CT - AT
        p.wt  = p.tat - p.bt           # Waiting Time    = TAT - BT

        gantt.append((f"P{p.pid}", start, current_time))

    return procs, gantt


# ─────────────────────────────────────────
# TASK 3: SJF Scheduling (Non-Preemptive)
# ─────────────────────────────────────────

def sjf_scheduling(processes):
    """
    Shortest Job First - Non-Preemptive.

    Algorithm:
      1. At each scheduling point, build the ready queue of
         processes that have arrived and are not yet completed.
      2. Select the process with the smallest Burst Time.
         (Ties are broken by Arrival Time.)
      3. Execute it to completion; update metrics.
      4. If no process is ready, advance time to next arrival (CPU idle).
      5. Repeat until all processes complete.

    Returns:
      procs (list)  : Processes with computed metrics.
      gantt (list)  : Gantt chart data as (label, start, end) tuples.
    """
    procs = copy.deepcopy(processes)
    n = len(procs)
    completed = [False] * n
    current_time = 0
    gantt = []
    done = 0

    while done < n:
        # Build ready queue: processes that have arrived and are not done
        ready = [
            procs[i] for i in range(n)
            if not completed[i] and procs[i].at <= current_time
        ]

        if not ready:
            # CPU is idle — jump forward to the next process arrival
            next_arrival = min(procs[i].at for i in range(n) if not completed[i])
            gantt.append(("IDLE", current_time, next_arrival))
            current_time = next_arrival
            continue

        # Pick process with shortest burst time; break ties by arrival time
        selected = min(ready, key=lambda p: (p.bt, p.at))
        idx = next(i for i in range(n) if procs[i].pid == selected.pid)

        start = current_time
        current_time += selected.bt

        # Calculate scheduling metrics
        selected.ct  = current_time
        selected.tat = selected.ct - selected.at
        selected.wt  = selected.tat - selected.bt

        completed[idx] = True
        done += 1
        gantt.append((f"P{selected.pid}", start, current_time))

    return procs, gantt


# ─────────────────────────────────────────
# TASK 4: Gantt Chart (Text-based)
# ─────────────────────────────────────────

def print_gantt_chart(gantt, title):
    """
    Draws a text-based Gantt chart on the terminal.

    Format:
      +--------+------+------+
      |   P1   |  P2  |  P3  |
      +--------+------+------+
      0        6     12     19
    """
    print(f"\n  Gantt Chart — {title}")

    # Build the top/bottom border string
    bar = ""
    widths = []
    for label, start, end in gantt:
        w = max((end - start) * 2, len(label) + 2)
        widths.append(w)
        bar += "+" + "-" * w
    bar += "+"

    print("  " + bar)

    # Process label row
    print("  ", end="")
    for i, (label, start, end) in enumerate(gantt):
        print(f"|{label:^{widths[i]}}", end="")
    print("|")

    print("  " + bar)

    # Time markers row
    print("  ", end="")
    times = [gantt[0][1]] + [end for _, _, end in gantt]
    for i, (label, start, end) in enumerate(gantt):
        t_str = str(times[i])
        print(t_str, end="")
        padding = widths[i] + 1 - len(t_str)
        print(" " * padding, end="")
    print(times[-1])
    print()


# ─────────────────────────────────────────
# Results Display
# ─────────────────────────────────────────

def display_results(procs, title):
    """
    Display scheduling results in a formatted table and compute averages.

    Columns: PID | AT | BT | CT | TAT | WT
    """
    print(f"\n{'='*65}")
    print(f"   {title} — Scheduling Results")
    print(f"{'='*65}")
    print(f"  {'PID':<6} {'AT':<6} {'BT':<6} {'CT':<6} {'TAT':<8} {'WT':<6}")
    print("  " + "-"*55)
    for p in procs:
        print(f"  P{p.pid:<5} {p.at:<6} {p.bt:<6} {p.ct:<6} {p.tat:<8} {p.wt:<6}")
    print("=" * 65)

    avg_tat = sum(p.tat for p in procs) / len(procs)
    avg_wt  = sum(p.wt  for p in procs) / len(procs)
    print(f"  Average Turnaround Time : {avg_tat:.2f}")
    print(f"  Average Waiting Time    : {avg_wt:.2f}")
    print("=" * 65)
    return avg_tat, avg_wt


# ─────────────────────────────────────────
# TASK 5: Performance Analysis & Comparison
# ─────────────────────────────────────────

def compare_algorithms(fcfs_tat, fcfs_wt, sjf_tat, sjf_wt):
    """
    Compare FCFS and SJF based on average TAT and WT.
    Determine which algorithm performs better and explain why.
    """
    print("\n" + "=" * 55)
    print("   TASK 5: Performance Comparison")
    print("=" * 55)
    print(f"  {'Metric':<30} {'FCFS':>8} {'SJF':>8}")
    print("  " + "-" * 45)
    print(f"  {'Avg Turnaround Time':<30} {fcfs_tat:>8.2f} {sjf_tat:>8.2f}")
    print(f"  {'Avg Waiting Time':<30} {fcfs_wt:>8.2f} {sjf_wt:>8.2f}")
    print("=" * 55)

    # Determine which is better based on average waiting time
    better = "SJF" if sjf_wt < fcfs_wt else "FCFS"
    wt_improvement = abs(fcfs_wt - sjf_wt)

    print(f"\n  ✅ Better Algorithm: {better}")
    print(f"  📉 WT Improvement  : {wt_improvement:.2f} units\n")

    print("  Analysis:")
    print("  ─────────────────────────────────────────────────")
    print("  FCFS:")
    print("    • Simple and easy to implement.")
    print("    • Fair — processes execute in arrival order.")
    print("    • Can cause 'Convoy Effect': short jobs stuck")
    print("      behind long ones, increasing average WT.")
    print("    • No starvation.")
    print()
    print("  SJF (Non-Preemptive):")
    print("    • Optimal average waiting time (proven).")
    print("    • Reduces convoy effect by prioritizing short jobs.")
    print("    • Risk of starvation for long processes.")
    print("    • Requires knowing burst times in advance.")
    print()
    print("  Conclusion:")
    print("    SJF is more CPU-efficient (lower avg WT & TAT),")
    print("    ideal for batch processing systems. FCFS is")
    print("    preferred when simplicity and fairness are key.")
    print("=" * 55)


# ─────────────────────────────────────────
# MAIN — Entry Point
# ─────────────────────────────────────────

def main():
    print("\n" + "★" * 55)
    print("  OS Lab Assignment 1 — CPU Scheduling Algorithms")
    print("  FCFS & SJF  |  ENCA252  |  BCA (AI & DS)")
    print("★" * 55)

    # Input selection
    print("\n  [1] Use default sample processes (recommended)")
    print("  [2] Enter your own processes")
    choice = input("\n  Your choice (1/2): ").strip()

    if choice == "2":
        processes = get_processes_from_user()
    else:
        processes = get_default_processes()
        print("\n  Using default sample processes.")

    # ── Task 1: Display input ──
    display_input_table(processes)

    # ── Task 2: FCFS ──
    print("\n" + "─" * 55)
    print("  TASK 2: FCFS Scheduling (Non-Preemptive)")
    print("─" * 55)
    fcfs_procs, fcfs_gantt = fcfs_scheduling(processes)
    print_gantt_chart(fcfs_gantt, "FCFS")
    fcfs_avg_tat, fcfs_avg_wt = display_results(fcfs_procs, "FCFS")

    # ── Task 3: SJF ──
    print("\n" + "─" * 55)
    print("  TASK 3: SJF Scheduling (Non-Preemptive)")
    print("─" * 55)
    sjf_procs, sjf_gantt = sjf_scheduling(processes)
    print_gantt_chart(sjf_gantt, "SJF")
    sjf_avg_tat, sjf_avg_wt = display_results(sjf_procs, "SJF")

    # ── Task 5: Comparison ──
    compare_algorithms(fcfs_avg_tat, fcfs_avg_wt, sjf_avg_tat, sjf_avg_wt)

    print("\n  ✅ Assignment Complete!\n")


if __name__ == "__main__":
    main()


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  OS Lab Assignment 1 — CPU Scheduling Algorithms
  FCFS & SJF  |  ENCA252  |  BCA (AI & DS)
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

  [1] Use default sample processes (recommended)
  [2] Enter your own processes



  Your choice (1/2):  1



  Using default sample processes.

   Process Input Table
PID      Arrival Time     Burst Time
----------------------------------------
P1       0                6
P2       1                8
P3       2                7
P4       3                3
P5       4                4

───────────────────────────────────────────────────────
  TASK 2: FCFS Scheduling (Non-Preemptive)
───────────────────────────────────────────────────────

  Gantt Chart — FCFS
  +------------+----------------+--------------+------+--------+
  |     P1     |       P2       |      P3      |  P4  |   P5   |
  +------------+----------------+--------------+------+--------+
  0            6                14             21     24       28


   FCFS — Scheduling Results
  PID    AT     BT     CT     TAT      WT    
  -------------------------------------------------------
  P1     0      6      6      6        0     
  P2     1      8      14     13       5     
  P3     2      7      21     19       12    
  P4     3 